# Bangkok PM2.5 Forecasting — Step 3: Model Training

**Models**: ST-UNN (primary) + LSTM / GRU / MLP / Persistence (baselines)  
**Input**: Preprocessed data from `data/gold/model_ready/`  
**Output**: Trained models, evaluation metrics, SHAP analysis

### Models
| Model | Type | Description |
|-------|------|-------------|
| **Persistence** | Naive baseline | Predict PM2.5(t) = PM2.5(t-1) |
| **MLP** | Feed-forward | Flattened sequence → FC layers |
| **LSTM** | Recurrent | Standard LSTM encoder → FC head |
| **GRU** | Recurrent | GRU encoder → FC head |
| **ST-UNN** | Spatio-Temporal | Temporal encoder + Spatial attention + Feature fusion |

### Evaluation
- MAE (primary), RMSE (secondary)
- Walk-forward validation
- Seasonal breakdown
- Extreme pollution day accuracy (PM2.5 > 50 µg/m³)

In [ ]:
import platform, subprocess, sys

def _install_if_missing(packages: list[str]) -> None:
    """Install only packages that are not already importable."""
    missing = []
    import_map = {
        "scikit-learn": "sklearn",
        "pyarrow": "pyarrow",
        "geopandas": "geopandas",
        "ipykernel": "ipykernel",
    }
    for pkg in packages:
        mod = import_map.get(pkg, pkg.split("[")[0])
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    else:
        print("All packages already installed.")

_install_if_missing([
    "pandas", "pyarrow", "numpy", "requests",
    "matplotlib", "seaborn", "scikit-learn",
    "geopandas", "xarray", "shap",
])

# ── PyTorch: auto-detect backend (ROCm / MPS / CUDA / CPU) ──
try:
    import torch
    hip = getattr(torch.version, "hip", None)
    cuda_ver = getattr(torch.version, "cuda", None)
    has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

    if hip:
        print(f"torch {torch.__version__} — AMD ROCm ({hip})")
    elif cuda_ver and torch.cuda.is_available():
        print(f"torch {torch.__version__} — NVIDIA CUDA ({cuda_ver})")
    elif has_mps:
        print(f"torch {torch.__version__} — Apple MPS (Metal Performance Shaders)")
    else:
        print(f"torch {torch.__version__} — CPU only")
except ImportError:
    IS_MAC = platform.system() == "Darwin"
    if IS_MAC:
        print("torch not found — install via: pip install torch torchvision torchaudio")
    else:
        print("torch not found — install ROCm wheels from repo.radeon.com")
        print("  or: pip install torch torchvision torchaudio")

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

# ── Make sure project root is on sys.path ────────────────────────────────
_NB_CWD = Path.cwd()
for _c in [_NB_CWD, *_NB_CWD.parents]:
    if (_c / "pyproject.toml").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

# ── Import everything needed from src.training ───────────────────────────
from src.training import (
    CombinedLoss,
    GRUForecaster,
    LSTMForecaster,
    MLPForecaster,
    PM25SequenceDataset,
    PersistenceModel,
    STUNN,
    TrainConfig,
    Trainer,
    build_model_catalog,
    compute_metrics,
    compute_seasonal_metrics,
    compute_extreme_accuracy,
    count_parameters,
    load_deployment_bundle,
    load_all_checkpoints,
    seed_everything,
    select_device,
)

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
seed_everything(SEED)
DEVICE = select_device()

print(f"Random seed  : {SEED}  (Python, NumPy, PyTorch all seeded)")
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"Platform     : {platform.system()} {platform.machine()}")


---
## 1. Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
CFG = TrainConfig(seed=SEED)
CFG.output_dir.mkdir(parents=True, exist_ok=True)


class _NbLog:
    """Notebook-safe logger (works in both Jupyter and nbconvert contexts)."""
    def _fmt(self, level, event, **kw):
        kw_str = "  ".join(f"{k}={v}" for k, v in kw.items())
        print(f"[{level:7s}] {event}  {kw_str}".rstrip(), flush=True)

    def info(self, event, **kw):    self._fmt("INFO",    event, **kw)
    def debug(self, event, **kw):   self._fmt("DEBUG",   event, **kw)
    def warning(self, event, **kw): self._fmt("WARNING", event, **kw)
    def error(self, event, **kw):   self._fmt("ERROR",   event, **kw)


LOG = _NbLog()
LOG.info("config.loaded",
         hidden_size=CFG.hidden_size, num_layers=CFG.num_layers,
         lr=CFG.learning_rate, epochs=CFG.epochs, seed=CFG.seed)


---
## 2. Load Preprocessed Data

In [ ]:
def load_manifest(data_dir: Path) -> dict:
    """Load and validate pipeline manifest for feature/target metadata."""
    manifest_path = data_dir / "pipeline_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")
    if not manifest_path.is_file():
        raise ValueError(f"Manifest path is not a file: {manifest_path}")
    with open(manifest_path) as f:
        manifest = json.load(f)
    required_keys = ["feature_cols", "target_col", "sequence_length", "forecast_horizons"]
    missing = [k for k in required_keys if k not in manifest]
    if missing:
        raise ValueError(f"Manifest missing required keys: {missing}")
    return manifest


MANIFEST = load_manifest(CFG.data_dir)
# Support both preprocessing_pipeline format (feature_cols, target_col, ...) and legacy (features.columns)
FEATURE_COLS = MANIFEST.get("feature_cols") or (MANIFEST.get("features") or {}).get("columns") or []
FEATURE_COLS = [c for c in FEATURE_COLS if c not in ("date", "stationID", "lat", "lon", "split")]
TARGET_COL = MANIFEST.get("target_col") or "pm2_5_ugm3"
SEQ_LEN = MANIFEST.get("sequence_length") or 30
HORIZONS = MANIFEST.get("forecast_horizons") or [1, 3]
NUM_FEATURES = MANIFEST.get("num_features") or len(FEATURE_COLS)

print(f"Features     : {NUM_FEATURES}")
print(f"Sequence len : {SEQ_LEN} days")
print(f"Horizons     : {HORIZONS} days")
print(f"Target       : {TARGET_COL}")
hotspot_in = [c for c in FEATURE_COLS if c.startswith("hotspot")]
print(f"Hotspot features : {len(hotspot_in)}")


In [ ]:
# ── Train/Val/Test split ratios (chronological) ─────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO = 1 - TRAIN_RATIO - VAL_RATIO = 0.15 (implicit)


def load_all_data(data_dir):
    """Load train.parquet, drop rows with any NaN, sort chronologically.

    val.parquet and test.parquet may contain future dates with missing features;
    we re-split from train.parquet using chronological 70/15/15 ratios.
    """
    from pathlib import Path
    import polars as pl
    train_path = Path(data_dir) / "train.parquet"
    if not train_path.exists():
        raise FileNotFoundError(f"Training data not found: {train_path}")
    df = pl.read_parquet(train_path)
    df = df.drop_nulls()
    df = df.sort(["date", "stationID"])
    LOG.info("data.loaded", rows=len(df), path=str(train_path))
    return df


def chronological_split(df, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO):
    """Split a Polars DataFrame chronologically by unique dates (no leakage)."""
    import polars as pl
    unique_dates = df["date"].unique().sort()
    n = len(unique_dates)
    tr_end = unique_dates[int(n * train_ratio)]
    va_end = unique_dates[min(int(n * (train_ratio + val_ratio)), n - 1)]
    df_tr = df.filter(pl.col("date") < tr_end)
    df_va = df.filter((pl.col("date") >= tr_end) & (pl.col("date") < va_end))
    df_te = df.filter(pl.col("date") >= va_end)
    LOG.info("split.done",
             train=len(df_tr), val=len(df_va), test=len(df_te),
             train_end=str(tr_end), val_end=str(va_end))
    return df_tr, df_va, df_te


_all_data = load_all_data(CFG.data_dir)
df_train, df_val, df_test = chronological_split(_all_data)

# PM25SequenceDataset is imported from src.training (with in-memory caching).
ds_train = PM25SequenceDataset(df_train, FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS, cache=True)
ds_val   = PM25SequenceDataset(df_val,   FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS, cache=True)
ds_test  = PM25SequenceDataset(df_test,  FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS, cache=True)

loader_train = DataLoader(ds_train, batch_size=CFG.batch_size, shuffle=True,  num_workers=0, pin_memory=DEVICE.type == "cuda")
loader_val   = DataLoader(ds_val,   batch_size=CFG.batch_size, shuffle=False, num_workers=0)
loader_test  = DataLoader(ds_test,  batch_size=CFG.batch_size, shuffle=False, num_workers=0)

LOG.info("sequences.created", train=len(ds_train), val=len(ds_val), test=len(ds_test))

if len(ds_train) > 0:
    x0, y0 = ds_train[0]
    print(f"Input shape : {x0.shape}")
    print(f"Target shape: {y0.shape}")
else:
    print("WARNING: 0 training sequences — PM2.5 data likely missing. Models will not train.")


---
## 3. Loss Function

Combined MAE + RMSE loss as specified in ST-UNN design.

In [ ]:
# CombinedLoss is imported from src.training.loss (already imported in cell 2).
# The criterion uses mae_weight=CFG.mae_weight, rmse_weight=CFG.rmse_weight.
criterion = CombinedLoss(mae_weight=CFG.mae_weight, rmse_weight=CFG.rmse_weight)
print(f"Loss fn : {criterion}")


---
## 4. Model Definitions

### 4a. Persistence Baseline

In [ ]:
# PersistenceModel is imported from src.training.models (already imported in cell 2).
# Usage: PersistenceModel(pm25_feature_idx, num_horizons)
print("PersistenceModel : OK (imported from src.training)")


### 4b. MLP Baseline

In [ ]:
# MLPForecaster is imported from src.training.models (already imported in cell 2).
# Note: first hidden layer = CFG.hidden_size * CFG.mlp_hidden_multiplier (default → 256).
print("MLPForecaster    : OK (imported from src.training)")


### 4c. LSTM Forecaster

In [ ]:
# LSTMForecaster is imported from src.training.models (already imported in cell 2).
print("LSTMForecaster   : OK (imported from src.training)")


### 4d. GRU Forecaster

In [ ]:
# GRUForecaster is imported from src.training.models (already imported in cell 2).
print("GRUForecaster    : OK (imported from src.training)")


### 4e. ST-UNN (Spatio-Temporal Unified Neural Network)

Architecture per `st-unn.mdc`:  
1. **Temporal Encoder** — GRU captures sequential dependencies  
2. **Spatial Attention** — Multi-head attention weights feature importance (upwind, hotspot, etc.)  
3. **Feature Fusion** — Gated fusion of temporal and spatial representations  
4. **Regression Head** — FC layers output PM2.5 at each forecast horizon

In [ ]:
# STUNN (with SpatialAttention + GatedFusion) is imported from src.training.models.
# Supports gradient checkpointing via STUNN(..., use_checkpointing=True).
print("STUNN            : OK (imported from src.training)")


---
## 5. Training Engine

In [ ]:
# Training helpers (train_one_epoch, evaluate) are encapsulated in src.training.trainer.Trainer.
# Trainer handles AMP, gradient accumulation, gradient clipping, early stopping, LR scheduling.
print("Trainer          : OK (imported from src.training)")


In [ ]:
def train_model(model, loader_train, loader_val, cfg, model_name):
    """Thin wrapper around src.training.Trainer for notebook compatibility."""
    trainer = Trainer(
        model=model,
        cfg=cfg,
        model_name=model_name,
    )
    history = trainer.train(loader_train, loader_val)
    return history


def evaluate(model, loader, _criterion):
    """Evaluate a model on a DataLoader, return (total_loss, preds_np, targets_np)."""
    import torch
    model.eval()
    preds_list, targets_list, total_loss = [], [], 0.0
    with torch.no_grad():
        for x_b, y_b in loader:
            x_b = x_b.to(DEVICE)
            out = model(x_b).cpu()
            loss_val = _criterion(out, y_b).item()
            total_loss += loss_val * len(x_b)
            preds_list.append(out.numpy())
            targets_list.append(y_b.numpy())
    import numpy as _np
    preds   = _np.concatenate(preds_list)   if preds_list   else _np.empty((0, len(HORIZONS)))
    targets = _np.concatenate(targets_list) if targets_list else _np.empty((0, len(HORIZONS)))
    return total_loss / max(1, len(preds)), preds, targets


# pm25_lag1_idx: index of pm2_5_ugm3_lag1 feature (used by PersistenceModel)
pm25_lag1_idx = FEATURE_COLS.index("pm2_5_ugm3_lag1") if "pm2_5_ugm3_lag1" in FEATURE_COLS else None
print(f"pm25_lag1_idx: {pm25_lag1_idx}")


---
## 6. Metrics

In [ ]:
# compute_metrics is imported from src.training.evaluator (already imported in cell 2).
# Returns a Polars DataFrame with columns: horizon_days, MAE, RMSE, R2.
print("compute_metrics  : OK (imported from src.training)")


---
## 7. Train All Models

In [ ]:
NUM_HORIZONS = len(HORIZONS)


def build_all_models():
    """Build all model architectures via src.training.build_model_catalog."""
    catalog = build_model_catalog(
        num_features=NUM_FEATURES,
        seq_len=SEQ_LEN,
        num_horizons=NUM_HORIZONS,
        hidden_size=CFG.hidden_size,
        num_layers=CFG.num_layers,
        dropout=CFG.dropout,
        attention_heads=CFG.attention_heads,
        mlp_hidden_multiplier=CFG.mlp_hidden_multiplier,
    )
    print("Model catalog:")
    for name, m in catalog.items():
        print(f"  {name:8s}: {count_parameters(m):>10,} trainable parameters")
    return catalog


# ── Checkpoint-first: load from disk if all four best-checkpoints exist ───
from src.training.inference import load_all_checkpoints as _load_ckpts

_ckpt_files = {name: CFG.output_dir / f"{name}_best.pt"
               for name in ["MLP", "LSTM", "GRU", "ST-UNN"]}
_all_ckpts_exist = all(p.exists() for p in _ckpt_files.values())

histories = {}

if _all_ckpts_exist:
    print(">>> All checkpoints found — loading from disk (skipping training). <<<")
    for name, path in _ckpt_files.items():
        print(f"  {name:8s}: {path}")
    models = _load_ckpts(
        models_dir=CFG.output_dir,
        num_features=NUM_FEATURES,
        seq_len=SEQ_LEN,
        num_horizons=NUM_HORIZONS,
        cfg=CFG,
        device=DEVICE,
    )
    print(f"\nLoaded: {list(models.keys())}")
elif len(ds_train) == 0:
    print("Cannot train — no valid sequences. PM2.5 data is required.")
    models = build_all_models()
else:
    print(">>> No checkpoints found — training from scratch. <<<")
    models = build_all_models()
    for name, model in models.items():
        print(f"\n{'='*50}\n  Training: {name}\n{'='*50}")
        histories[name] = train_model(model, loader_train, loader_val, CFG, name)


---
## 8. Evaluate on Test Set

In [ ]:
results = {}

if len(ds_test) > 0:
    # Persistence baseline
    if pm25_lag1_idx is not None:
        persist = PersistenceModel(pm25_lag1_idx, NUM_HORIZONS)
        p_preds, p_targets = [], []
        for x_batch, y_batch in loader_test:
            p_preds.append(persist.predict(x_batch).numpy())
            p_targets.append(y_batch.numpy())
        results["Persistence"] = {
            "preds": np.concatenate(p_preds),
            "targets": np.concatenate(p_targets),
        }

    # Neural models
    for name, model in models.items():
        model = model.to(DEVICE)
        _, preds, targets = evaluate(model, loader_test, criterion)
        results[name] = {"preds": preds, "targets": targets}

    # Metrics comparison — compute_metrics returns a Polars DataFrame
    import polars as _pl
    all_metrics_pl = []
    for name, res in results.items():
        df_m = compute_metrics(res["preds"], res["targets"], HORIZONS)
        df_m = df_m.with_columns(_pl.lit(name).alias("model"))
        all_metrics_pl.append(df_m)

    metrics_df = _pl.concat(all_metrics_pl)
    metrics_pd  = metrics_df.to_pandas()

    print("\n" + "=" * 60)
    print("  TEST SET METRICS")
    print("=" * 60)
    pivot = metrics_pd.pivot_table(index="model", columns="horizon_days", values=["MAE", "RMSE", "R2"])
    print(pivot.round(3).to_string())

    metrics_pd.to_csv(CFG.output_dir / "test_metrics.csv", index=False)
    print(f"\nSaved: {CFG.output_dir / 'test_metrics.csv'}")
else:
    print("No test sequences available — skipping evaluation.")
    metrics_df = None
    metrics_pd  = None


---
## 9. Seasonal & Extreme Day Analysis

In [ ]:
if results:
    # dates must be a Polars Series for compute_seasonal_metrics
    test_dates_pl = df_test.sort(["stationID", "date"])["date"]

    print("\n--- Seasonal Breakdown (ST-UNN) ---")
    if "ST-UNN" in results:
        seasonal = compute_seasonal_metrics(
            results["ST-UNN"]["preds"], results["ST-UNN"]["targets"],
            test_dates_pl, HORIZONS
        )
        print(seasonal.to_pandas().to_string(index=False))

    print("\n--- Extreme Pollution Day Detection ---")
    for name, res in results.items():
        ext = compute_extreme_accuracy(res["preds"], res["targets"])
        print(f"  {name:12s}: {ext}")
else:
    print("No results to analyze.")


---
## 10. Training Curves & Comparison Plots

In [ ]:
if histories:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    for name, h in histories.items():
        ax.plot(h["train_loss"], label=f"{name} (train)", alpha=0.7)
        ax.plot(h["val_loss"], label=f"{name} (val)", linestyle="--", alpha=0.7)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training & Validation Loss")
    ax.legend(fontsize=8)

    ax = axes[1]
    if results:
        model_names = list(results.keys())
        mae_24h = [compute_metrics(results[n]["preds"], results[n]["targets"], HORIZONS).iloc[0]["MAE"] for n in model_names]
        colors = sns.color_palette("viridis", len(model_names))
        bars = ax.bar(model_names, mae_24h, color=colors)
        ax.set_ylabel("MAE (µg/m³)")
        ax.set_title("24h Forecast — MAE Comparison")
        for bar, v in zip(bars, mae_24h):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{v:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training histories — skipping plots.")

In [ ]:
if results and "ST-UNN" in results:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(7 * len(HORIZONS), 6))
    if len(HORIZONS) == 1:
        axes = [axes]

    for i, (ax, h) in enumerate(zip(axes, HORIZONS)):
        p = results["ST-UNN"]["preds"][:, i]
        t = results["ST-UNN"]["targets"][:, i]
        ax.scatter(t, p, alpha=0.15, s=4, color="tab:blue")
        lims = [min(t.min(), p.min()), max(t.max(), p.max())]
        ax.plot(lims, lims, "r--", linewidth=1, label="Perfect")
        ax.set_xlabel(f"Actual PM2.5 (+{h}d)")
        ax.set_ylabel(f"Predicted PM2.5 (+{h}d)")
        ax.set_title(f"ST-UNN — {h}-Day Forecast")
        ax.legend()

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "stunn_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 11. SHAP Feature Importance (ST-UNN)

In [ ]:
if len(ds_test) > 0 and "ST-UNN" in models:
    try:
        import shap
    except ImportError:
        print("shap not installed — skipping. Run: pip install shap")
        shap = None

    if shap is not None:
        stunn_model = models["ST-UNN"].to(DEVICE).eval()

        shap_sample_size = min(200, len(ds_test))
        bg_size = min(50, len(ds_train))

        sample_x = torch.stack([ds_test[i][0] for i in range(shap_sample_size)]).to(DEVICE)
        bg_x     = torch.stack([ds_train[i][0] for i in range(bg_size)]).to(DEVICE)

        def model_predict_24h(x: torch.Tensor) -> torch.Tensor:
            """Return only the 24h horizon as a 1-D output (required by SHAP)."""
            with torch.no_grad():
                return stunn_model(x)[:, 0]   # shape (batch,) — NOT (batch, 1)

        explainer = shap.GradientExplainer(model_predict_24h, bg_x)
        shap_values = explainer.shap_values(sample_x)

        # Collapse to 1-D feature importance regardless of SHAP output shape
        if isinstance(shap_values, list):
            sv = np.array(shap_values[0])  # (n, seq, features)
        else:
            sv = np.array(shap_values)     # (n, seq, features) or (n, seq, features, 1)

        # Reduce all axes except the feature axis (axis=-1 after squeezing)
        sv = sv.squeeze()  # remove trailing size-1 dims
        if sv.ndim == 3:   # (n_samples, seq_len, n_features)
            mean_abs_shap = np.mean(np.abs(sv), axis=(0, 1))
        elif sv.ndim == 2: # (n_samples, n_features) — already collapsed
            mean_abs_shap = np.mean(np.abs(sv), axis=0)
        else:
            mean_abs_shap = np.abs(sv).flatten()

        mean_abs_shap = mean_abs_shap.flatten()   # guarantee 1-D

        feat_importance = pd.DataFrame({
            "feature": FEATURE_COLS[:len(mean_abs_shap)],
            "mean_abs_shap": mean_abs_shap,
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

        print("\n--- Top 15 Features (SHAP) ---")
        print(feat_importance.head(15).to_string(index=False))
        feat_importance.to_csv(CFG.output_dir / "feature_importance_shap.csv", index=False)

        fig, ax = plt.subplots(figsize=(10, 8))
        top20 = feat_importance.head(20)
        ax.barh(top20["feature"][::-1], top20["mean_abs_shap"][::-1], color="steelblue")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title("ST-UNN Feature Importance (24h Forecast)")
        plt.tight_layout()
        plt.savefig(CFG.output_dir / "shap_importance.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("SHAP analysis requires test sequences + trained ST-UNN. Skipping.")


---
## 12. Export Best Model for Deployment

In [ ]:
if len(ds_train) > 0 and "ST-UNN" in models:
    export_path = CFG.output_dir / "stunn_deployment.pt"

    deployment_bundle = {
        "model_state_dict": models["ST-UNN"].cpu().state_dict(),
        "config": {
            "num_features": NUM_FEATURES,
            "hidden_size": CFG.hidden_size,
            "num_layers": CFG.num_layers,
            "num_horizons": NUM_HORIZONS,
            "attention_heads": CFG.attention_heads,
            "dropout": CFG.dropout,
            "seq_len": SEQ_LEN,
        },
        "manifest": MANIFEST,
        "feature_cols": FEATURE_COLS,
        "horizons": HORIZONS,
    }

    torch.save(deployment_bundle, export_path)
    LOG.info("Deployment bundle saved → %s (%.1f MB)", export_path, export_path.stat().st_size / 1e6)
else:
    print("No trained model to export.")

---
## 13. Forecasting with Trained Models

Generate predictions from all trained models on the test set, visualize forecast quality,
and provide a reusable `forecast()` function for deployment.

In [ ]:
@torch.no_grad()
def forecast_all_models(
    models_dict: dict[str, nn.Module],
    loader: DataLoader,
    horizons: list[int],
    persist_model: PersistenceModel | None = None,
) -> dict[str, dict[str, np.ndarray]]:
    """Run inference on all models, return {name: {preds, targets}} per model."""
    all_results: dict[str, dict[str, np.ndarray]] = {}

    # Persistence baseline
    if persist_model is not None:
        p_preds, p_targets = [], []
        for x_b, y_b in loader:
            p_preds.append(persist_model.predict(x_b).numpy())
            p_targets.append(y_b.numpy())
        all_results["Persistence"] = {
            "preds": np.concatenate(p_preds),
            "targets": np.concatenate(p_targets),
        }

    # Neural models
    for name, model in models_dict.items():
        model = model.to(DEVICE).eval()
        m_preds, m_targets = [], []
        for x_b, y_b in loader:
            pred = model(x_b.to(DEVICE)).cpu().numpy()
            m_preds.append(pred)
            m_targets.append(y_b.numpy())
        all_results[name] = {
            "preds": np.concatenate(m_preds),
            "targets": np.concatenate(m_targets),
        }

    return all_results


if len(ds_test) > 0:
    forecast_results = forecast_all_models(
        models, loader_test, HORIZONS,
        persist_model=PersistenceModel(pm25_lag1_idx, NUM_HORIZONS) if pm25_lag1_idx is not None else None,
    )
    LOG.info("Forecast generated for %d models, %d test samples", len(forecast_results), len(ds_test))
else:
    forecast_results = {}
    print("No test sequences — forecasting requires PM2.5 data.")

### 13a. Forecast Time Series — Actual vs Predicted

In [ ]:
if forecast_results:
    best_model_name = "ST-UNN" if "ST-UNN" in forecast_results else list(forecast_results.keys())[-1]

    for h_idx, h_days in enumerate(HORIZONS):
        fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={"height_ratios": [3, 1]})

        actual = forecast_results[best_model_name]["targets"][:, h_idx]
        n_plot = min(500, len(actual))
        x_axis = np.arange(n_plot)

        # Top: time-series overlay
        ax = axes[0]
        ax.plot(x_axis, actual[:n_plot], color="black", linewidth=1, alpha=0.8, label="Actual")

        model_colors = {"Persistence": "#9E9E9E", "MLP": "#FF9800", "LSTM": "#2196F3", "GRU": "#4CAF50", "ST-UNN": "#F44336"}
        for name, res in forecast_results.items():
            pred = res["preds"][:n_plot, h_idx]
            color = model_colors.get(name, "gray")
            lw = 1.5 if name == best_model_name else 0.8
            alpha = 0.9 if name == best_model_name else 0.5
            ax.plot(x_axis, pred, color=color, linewidth=lw, alpha=alpha, label=name)

        ax.axhline(50, color="red", linestyle=":", linewidth=0.8, alpha=0.4, label="Unhealthy (50 µg/m³)")
        ax.set_ylabel("PM2.5 (µg/m³)")
        ax.set_title(f"Forecast vs Actual — +{h_days} Day Horizon (first {n_plot} samples)")
        ax.legend(fontsize=8, ncol=3, loc="upper right")

        # Bottom: residuals for best model
        ax = axes[1]
        residuals = forecast_results[best_model_name]["preds"][:n_plot, h_idx] - actual[:n_plot]
        ax.bar(x_axis, residuals, width=1, color=np.where(residuals > 0, "#F44336", "#2196F3"), alpha=0.6)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylabel("Residual")
        ax.set_xlabel("Sample index")
        ax.set_title(f"{best_model_name} Residuals — mean={residuals.mean():.2f}, std={residuals.std():.2f}")

        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_timeseries_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13b. Error Distribution per Model

In [ ]:
if forecast_results:
    n_models = len(forecast_results)

    for h_idx, h_days in enumerate(HORIZONS):
        fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 5), sharey=True)
        if n_models == 1:
            axes = [axes]

        for i, (name, res) in enumerate(forecast_results.items()):
            ax = axes[i]
            errors = res["preds"][:, h_idx] - res["targets"][:, h_idx]
            color = model_colors.get(name, "gray")

            ax.hist(errors, bins=60, color=color, alpha=0.7, edgecolor="white", density=True)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.axvline(errors.mean(), color="red", linestyle="--", linewidth=1,
                       label=f"bias={errors.mean():.2f}")
            ax.set_xlabel("Error (pred - actual)")
            ax.set_title(f"{name}\nMAE={np.abs(errors).mean():.2f}")
            ax.legend(fontsize=8)

        axes[0].set_ylabel("Density")
        fig.suptitle(f"Forecast Error Distribution — +{h_days} Day", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_error_dist_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13c. Rolling Forecast Accuracy (MAE over time)

In [ ]:
if forecast_results:
    ROLLING_WIN = 50

    for h_idx, h_days in enumerate(HORIZONS):
        fig, ax = plt.subplots(figsize=(16, 5))

        for name, res in forecast_results.items():
            abs_err = np.abs(res["preds"][:, h_idx] - res["targets"][:, h_idx])
            rolling_mae = pd.Series(abs_err).rolling(ROLLING_WIN, min_periods=1).mean()
            color = model_colors.get(name, "gray")
            lw = 2 if name == best_model_name else 1
            ax.plot(rolling_mae.values, color=color, linewidth=lw, alpha=0.8, label=name)

        ax.set_xlabel("Sample index")
        ax.set_ylabel(f"Rolling MAE (window={ROLLING_WIN})")
        ax.set_title(f"Rolling Forecast Accuracy — +{h_days} Day Horizon")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_rolling_mae_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13d. Forecast by PM2.5 Intensity Band

In [ ]:
if forecast_results:
    AQI_BANDS = [
        (0, 25, "Good (0-25)", "#4CAF50"),
        (25, 50, "Moderate (25-50)", "#FF9800"),
        (50, 100, "Unhealthy-Sensitive (50-100)", "#FF5722"),
        (100, float("inf"), "Unhealthy (100+)", "#F44336"),
    ]

    h_idx = 0
    h_days = HORIZONS[h_idx]
    actual = forecast_results[best_model_name]["targets"][:, h_idx]

    rows = []
    for name, res in forecast_results.items():
        pred = res["preds"][:, h_idx]
        for lo, hi, label, _ in AQI_BANDS:
            mask = (actual >= lo) & (actual < hi)
            if mask.sum() == 0:
                continue
            mae = np.abs(pred[mask] - actual[mask]).mean()
            rows.append({"model": name, "band": label, "MAE": mae, "n_samples": int(mask.sum())})

    band_df = pd.DataFrame(rows)

    if len(band_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        band_pivot = band_df.pivot(index="band", columns="model", values="MAE")
        band_pivot = band_pivot.reindex([b[2] for b in AQI_BANDS if b[2] in band_pivot.index])
        band_pivot.plot(kind="bar", ax=ax, width=0.75)
        ax.set_ylabel("MAE (µg/m³)")
        ax.set_title(f"Forecast MAE by PM2.5 Intensity Band — +{h_days} Day")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
        ax.legend(fontsize=8, title="Model")
        ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.savefig(CFG.output_dir / "forecast_by_aqi_band.png", dpi=150, bbox_inches="tight")
        plt.show()

        print(band_df.to_string(index=False))
else:
    print("No forecast results to plot.")

### 13e. Multi-Horizon Comparison (24h vs 72h)

In [ ]:
if forecast_results and len(HORIZONS) > 1:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(8 * len(HORIZONS), 6))
    if len(HORIZONS) == 1:
        axes = [axes]

    for h_idx, (ax, h_days) in enumerate(zip(axes, HORIZONS)):
        for name, res in forecast_results.items():
            pred = res["preds"][:, h_idx]
            actual = res["targets"][:, h_idx]
            color = model_colors.get(name, "gray")
            alpha = 0.25 if name == "Persistence" else 0.15
            s = 6 if name == best_model_name else 3
            ax.scatter(actual, pred, alpha=alpha, s=s, color=color, label=name)

        lims = [
            min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1]),
        ]
        ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5, label="Perfect")
        ax.set_xlabel(f"Actual PM2.5 (+{h_days}d)")
        ax.set_ylabel(f"Predicted PM2.5 (+{h_days}d)")
        ax.set_title(f"+{h_days} Day Horizon — All Models")
        ax.legend(fontsize=7, markerscale=3)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "forecast_multi_horizon_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
elif forecast_results:
    print("Single horizon — scatter already shown in section 10.")
else:
    print("No forecast results to plot.")

### 13f. Reusable `forecast()` Function for Deployment

Load a saved model bundle and run inference on new data.

In [ ]:
# load_deployment_bundle is imported from src.training.inference.
# It reconstructs STUNN from the config stored in the bundle and returns:
#   (model, config_dict, feature_cols, horizons)

def load_deployment_model(bundle_path, device="mps"):
    """Thin wrapper around src.training.load_deployment_bundle for notebook compat."""
    import torch
    dev = torch.device(device if torch.cuda.is_available() or
                       (device == "mps" and hasattr(torch.backends, "mps") and
                        torch.backends.mps.is_available()) else "cpu")
    model, cfg, feat_cols, horizons = load_deployment_bundle(bundle_path, device=dev)
    return model, {"config": cfg, "feature_cols": feat_cols, "horizons": horizons}


@torch.no_grad()
def forecast(model, input_sequence, feature_cols=None, horizons=None):
    """Run a single forecast from a (seq_len, num_features) array.

    Args:
        model: Trained model in eval mode.
        input_sequence: np.ndarray or Tensor of shape (seq_len, num_features).
        feature_cols: Column names (used only for display).
        horizons: Forecast horizons in days.

    Returns:
        pd.DataFrame with columns [horizon_days, pm25_forecast].
    """
    if isinstance(input_sequence, np.ndarray):
        input_sequence = torch.from_numpy(input_sequence).float()
    if input_sequence.dim() == 2:
        input_sequence = input_sequence.unsqueeze(0)
    device = next(model.parameters()).device
    pred = model(input_sequence.to(device)).cpu().numpy().squeeze()
    if horizons is None:
        horizons = list(range(1, len(pred) + 1))
    return pd.DataFrame({"horizon_days": horizons, "pm25_forecast": pred})


# Demo: run forecast on the last test sample
if len(ds_test) > 0 and "ST-UNN" in models:
    demo_x, demo_y = ds_test[-1]
    demo_result = forecast(models["ST-UNN"].to(DEVICE), demo_x, FEATURE_COLS, HORIZONS)
    print("--- Demo Forecast (last test sample) ---")
    print(demo_result.to_string(index=False))
    print(f"\nActual values: {demo_y.numpy()}")
else:
    # Try loading from deployment bundle
    bundle_path = CFG.output_dir / "stunn_deployment.pt"
    if bundle_path.exists():
        _m, _meta = load_deployment_model(bundle_path, device=str(DEVICE))
        print(f"Loaded deployment bundle: {_meta['config']['num_features']} features, "
              f"horizons={_meta['horizons']}")
    else:
        print("Demo forecast skipped — no trained model or test data.")


### 13g. Forecast Summary Table

In [ ]:
if forecast_results:
    summary_rows = []
    for name, res in forecast_results.items():
        for h_idx, h_days in enumerate(HORIZONS):
            p = res["preds"][:, h_idx]
            t = res["targets"][:, h_idx]
            mae = np.mean(np.abs(p - t))
            rmse = np.sqrt(np.mean((p - t) ** 2))
            bias = np.mean(p - t)
            ss_res = np.sum((t - p) ** 2)
            ss_tot = np.sum((t - np.mean(t)) ** 2)
            r2 = 1 - ss_res / (ss_tot + 1e-8)

            extreme_mask = t > 50
            extreme_mae = np.mean(np.abs(p[extreme_mask] - t[extreme_mask])) if extreme_mask.sum() > 0 else float("nan")

            summary_rows.append({
                "Model": name,
                "Horizon": f"+{h_days}d",
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "Bias": round(bias, 2),
                "R²": round(r2, 3),
                "Extreme MAE (>50)": round(extreme_mae, 2) if not np.isnan(extreme_mae) else "N/A",
                "N": len(t),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(CFG.output_dir / "forecast_summary.csv", index=False)

    print("=" * 80)
    print("  FORECAST PERFORMANCE SUMMARY")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    print(f"\nSaved → {CFG.output_dir / 'forecast_summary.csv'}")
else:
    print("No forecast results — PM2.5 data needed.")

---
## 14. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("  MODEL TRAINING — FINAL SUMMARY")
print("=" * 70)

print(f"\n--- Data ---")
print(f"  Train sequences: {len(ds_train):,}")
print(f"  Val sequences  : {len(ds_val):,}")
print(f"  Test sequences : {len(ds_test):,}")

if results:
    print(f"\n--- Best Test MAE (24h) ---")
    for name, res in results.items():
        m = compute_metrics(res["preds"], res["targets"], HORIZONS).to_pandas()
        row = m.iloc[0]
        print(f"  {name:12s}: MAE={row['MAE']:.2f}, RMSE={row['RMSE']:.2f}, R²={row['R2']:.3f}")

print(f"\n--- Outputs ---")
for fp in sorted(CFG.output_dir.iterdir()):
    if fp.suffix in (".pt", ".csv", ".png", ".json"):
        size_kb = fp.stat().st_size / 1024
        print(f"  {fp.name:<35s} {size_kb:6.1f} KB")
